In [2]:
import sys; sys.path.append("..")   # los notebooks están en notebooks/, src/ está un nivel arriba
from src.data import get_paths, load_split

paths = get_paths()
train = load_split("train")

/Users/lucatraverso/Documents/spanish-clinical-ner/spanish-clinical-ner/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import spacy
from spacy.tokens import Doc, DocBin
from src.data import read_conll
from results.evaluate import evaluate

nlp = spacy.blank("es")
for split in ["train", "dev"]:
    db = DocBin()
    for tokens, tags in read_conll(paths[split]):
        db.add(Doc(nlp.vocab, words=tokens, ents=tags))
    db.to_disk(f"../data/{split}.spacy")

In [5]:
!python -m spacy init config config.cfg --lang es --pipeline ner --optimize efficiency
!python -m spacy train config.cfg --output ../models/spacy --paths.train ../data/train.spacy --paths.dev ../data/dev.spacy




✘ The provided output file already exists. To force overwriting the
config file, set the --force or -F flag.

ℹ Saving to output directory: ../models/spacy
ℹ Using CPU
ℹ To switch to GPU 0, use the option: --gpu-id 0

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     49.33    0.00    0.00    0.00    0.00
  0     200         55.50   1159.56   23.01   32.91   17.68    0.23
  0     400         88.72    585.56   46.83   54.88   40.84    0.47
  0     600        481.14    578.13   57.44   63.84   52.21    0.57
  0     800        180.44    536.75   62.15   66.67   58.21    0.62
  0    1000        357.59    559.50   69.18   82.88   59.37    0.6

In [7]:
nlp = spacy.load("../models/spacy/model-best")
y_true, y_pred = [], []
for tokens, tags in read_conll(paths["test"]):
    doc = nlp(Doc(nlp.vocab, words=tokens))
    y_pred.append([f"{t.ent_iob_}-{t.ent_type_}" if t.ent_iob_ != "O" else "O"

                   for t in doc])
    y_true.append(tags)
evaluate(y_true, y_pred)

                  precision    recall  f1-score   support

   NORMALIZABLES     0.8974    0.7919    0.8413       961
NO_NORMALIZABLES     0.0000    0.0000    0.0000        10
       PROTEINAS     0.8780    0.7497    0.8088       835

       micro avg     0.8874    0.7680    0.8234      1806
       macro avg     0.5918    0.5139    0.5500      1806
    weighted avg     0.8835    0.7680    0.8216      1806



np.float64(0.8233897298901751)